In [1]:
import pricing_driven_service_allocation as pdsa
# URLS
PRIME_INSTANCE_URL = "http://localhost:3000/api/v1/"
# Paths
TOPOLOGIES_DIR = "synthetic-dataset/synthetic-topologies"
DEVICES_EUA_DATASET_PATH = "eua-dataset/edge-servers/site.csv"
CLIENTS_EUA_DATASET_PATH = "eua-dataset/users/users-aus.csv"
EXPERIMENT_CONFIGURATION_PATH = "experiments/experiment_test.yml"
# Scenario containers
topologies = {}
demands = {}
filters = {}
results = {}
# Configuration
SEED = 42
UNLIMITED_VALUE = 100000000
VENDORS_TO_CONSIDER = ["Telstra", "Optus", "Vodafone", "Telecom", "Macquarie"]
OFFER_CONFIGURATION = {
    'global': {
        'group_percentages': {1: 83, 2: 15, 3: 2},
        'group_ranges': {1: (0, 12.5), 2: (12.5, 50), 3: (50, 100)}
    },
    'attributes': {
        'available_RAM': {
            'min': 1,
            'max': 128,
            'default_price': 0.5,
            'price_by_provider_group': {
                'OPTUS': {1: 1, 2: 0.15, 3: 0.005},
                'TELSTRA': {1: 1.3, 2: 0.12, 3: 0.003},
                'VODAFONE': {1: 1.5, 2: 0.18, 3: 0.001},
                'MACQUARIE': {1: 0.2, 2: 0.17, 3: 0.008},
                'TELECOM': {1: 0.7, 2: 0.14, 3: 0.004},
            },
            'local_distribution': {
                1: [(70, 0, 25), (30, 25, 100)]
            }
        },
        'available_Storage': {
            'min': 1,
            'max': 100000,
            'default_price': 0.2,
            'price_by_provider_group': {
                'OPTUS': {1: 0.90, 2: 0.25, 3: 0.08},
                'TELSTRA': {1: 1.10, 2: 0.30, 3: 0.10},
                'VODAFONE': {1: 1.25, 2: 0.35, 3: 0.05},
                'MACQUARIE': {1: 0.50, 2: 0.20, 3: 0.09},
                'TELECOM': {1: 0.85, 2: 0.28, 3: 0.07},
            },
            'local_distribution': {
                1: [(70, 0, 0.016), (30, 0.512, 16.384)]
            }
        },
        'available_vCPU': {
            'min': 1,
            'max': 64,
            'default_price': 15
        },
        'available_GPU': {
            'min': 0,
            'max': 8,
            'default_price': 120,
            'price_by_provider_group': {
                'TELSTRA': {2: 130, 3: 150},
                'OPTUS': {3: 140}
            },
            'local_distribution': {
                1: [(85, 0, 25), (15, 25, 50)],
                2: [(60, 0, 50), (40, 50, 100)],
                3: [(30, 0, 50), (70, 50, 100)]
            }
        },
        'available_TPU': {
            'min': 0,
            'max': 4,
            'default_price': 200,
            'price_by_provider_group': {
                'TELSTRA': {3: 240},
                'OPTUS': {2: 210}
            },
            'local_distribution': {
                1: [(90, 0, 25), (10, 25, 50)],
                2: [(70, 0, 50), (30, 50, 100)],
                3: [(40, 0, 50), (60, 50, 100)]
            }
        }
    },
    'devices_types_by_group': {
        1: {'SENSOR': 25, 'CAMERA': 50, 'MOBILE': 25},
        2: {'LAPTOP': 25, 'COMPUTER': 50, 'MOBILE': 25},
        3: {'DATA_CENTER': 75, 'SERVER': 25}
    }
}

In [2]:
import yaml
import os

# Load the YAML file
with open(os.path.join(EXPERIMENT_CONFIGURATION_PATH), 'r') as file:
  config_data = yaml.safe_load(file)
  
for scenario_size in config_data['experiments'].keys():
    for app in config_data['experiments'][scenario_size].keys():
        topology_offer = config_data['experiments'][scenario_size][app]['offer']
        # Check that all providers in the topology offer are in the vendors to consider
        providers_in_offer = [p.lower() for p in topology_offer['providers'].keys()]
        vendors_lower = [v.lower() for v in VENDORS_TO_CONSIDER]

        if not all(provider in vendors_lower for provider in providers_in_offer):
          raise ValueError(f"Some providers in topology {scenario_size}/{app} offer are not in VENDORS_TO_CONSIDER."
                   f"Providers in offer: {providers_in_offer}, "
                   f"Vendors to consider: {vendors_lower}")

In [3]:
import os
import shutil

# Create all defined paths if they do not exist
paths_to_create = [
  TOPOLOGIES_DIR,
]

for path in paths_to_create:
  if path and not os.path.exists(path):
    os.makedirs(path, exist_ok=True)
    print(f"Created directory: {path}")
  elif path and os.path.exists(path):
    print(f"Directory already exists: {path}. Removing existing subfolders...")
    # Remove existing subfolders (including non-empty ones)
    for subfolder in os.listdir(path):
      subfolder_path = os.path.join(path, subfolder)
      if os.path.isdir(subfolder_path):
        shutil.rmtree(subfolder_path)
    print("Done!")
    

Directory already exists: synthetic-dataset/synthetic-topologies. Removing existing subfolders...
Done!


# 2. Filter EUA dataset

In this step, we will filter the EUA dataset to include only devices from which we can infer the vendor, usually throug their description. This is crucial for our analysis as it allows us to categorize devices and simulate exclusion and interoperability relationships among providers.

In addition, we will enrich the dataset with synthetic resources, unit prices, and device types. This enrichment is based on the OFFER_CONFIGURATION defined at the beginning of the notebook.

In [4]:
# Load dataset
devices_df = pdsa.dataset.load_devices_dataframe(DEVICES_EUA_DATASET_PATH)
# Filter devices from which vendor can be inferred
devices_df = pdsa.dataset.filter_devices_by_vendors(devices_df, VENDORS_TO_CONSIDER)
# Assign resources to devices based on the offer configuration
devices_df = pdsa.dataset.assign_device_resources(devices_df, OFFER_CONFIGURATION, SEED)

devices_df.head()

,latitude,longitude,elevation,provider,global_group,device_type,available_RAM,unit_price_available_RAM,available_Storage,unit_price_available_Storage,available_vCPU,unit_price_available_vCPU,available_GPU,unit_price_available_GPU,available_TPU,unit_price_available_TPU
device_id,,,,,,,,,,,,,,,,
10000002,-28.777660,114.634260,NaN,OPTUS,1,CAMERA,2,1.00,2,0.9,5,15.0,0,120.0,0,200.0
100001,-38.248652,144.605442,23.0,TELSTRA,2,MOBILE,1,0.12,1,0.3,7,15.0,0,130.0,0,200.0
10000114,-31.901910,152.533540,NaN,OPTUS,1,CAMERA,2,1.00,1,0.9,1,15.0,0,120.0,0,200.0
100002,-37.728550,145.222007,116.0,OPTUS,1,CAMERA,4,1.00,1,0.9,4,15.0,0,120.0,0,200.0
10000215,-32.981570,121.644400,NaN,TELSTRA,1,CAMERA,4,1.30,1,1.1,8,15.0,0,120.0,0,200.0


# 3. Topology Generation

In [5]:
for scenario_size in config_data['experiments'].keys():
    for app in config_data['experiments'][scenario_size].keys():
        topology_offer = config_data['experiments'][scenario_size][app]['offer']
        _, topology_id = pdsa.generators.topology(
                          lat = topology_offer['zone']['lat'],
                          long = topology_offer['zone']['long'],
                          center_elevation = topology_offer['zone'].get('elevation', 0),
                          rad=topology_offer['zone']['radius'],
                          devices_df=devices_df,
                          topologies_result_dir=TOPOLOGIES_DIR,
                          number_of_providers=len(topology_offer["providers"]),
                          allowed_groups=[1, 2, 3],
                          number_of_devices=topology_offer['max_devices'],
                          options={"seed": SEED, "logs": False}
                      )
        topologies[f"{scenario_size}_{app}"] = topology_id
        
print("---- Topologies generated ----\n")
for key, value in topologies.items():
  print(f"{key}: {value}")

---- Topologies generated ----

small_ar/vr: 41cf9dd2-00a1-4e48-8931-83e6a6410790
medium_ar/vr: b711f2ec-0bee-49b7-99c0-6a918fd2af1b
large_ar/vr: ac433fdf-4008-429a-8a37-3367b3543195


# 4. Problem instance generation

In [6]:
import os

if not os.path.exists("./iPricing/model"):
    os.makedirs("./iPricing/model")

!protoc --python_out=./iPricing/model ./iPricing/iPricing.proto
!mv ./iPricing/model/iPricing/iPricing_pb2.py ./iPricing/model/iPricing_pb2.py
!rm -rf ./iPricing/model/iPricing

In [7]:
import pandas as pd
from pricing_driven_service_allocation.generators.client_demand import AppType

c_locations_df = pd.read_csv(os.path.join(CLIENTS_EUA_DATASET_PATH), index_col=0)
n_clients = len(c_locations_df)

app_mapping = {
  "ar/vr": AppType.AR_VR,
  "video_privacy": AppType.VIDEO_PRIVACY,
  "lidar": AppType.LIDAR,
  "robot_iot": AppType.ROBOT_IOT
}

In [12]:

from iPricing.model.iPricing_pb2 import Pricing

for id, topology_id in topologies.items():
  scenario_size, app = id.split("_")
  topology_demand = config_data['experiments'][scenario_size][app]['demand']
  topology_request = config_data['experiments'][scenario_size][app]['request']
  
  users_demand = pdsa.generators.client_demand.calculate_resources(n_clients, app_mapping[app], topology_demand['concurrency'])
  
  print(f"---- Generating problem instance for scenario {id} ----")
  print("----- Demand -----")
  print(users_demand)
  
  request = {
              'currency': 'USD',
              'users_location': topology_demand['zone'],
              'providers_to_consider': topology_request['providers_to_consider'],
              'budget': topology_request['budget'],
              'max_devices': topology_request['max_devices'],
              'device_types': topology_request['devices_types_required'],
              'resources': {
                  'available_ram': users_demand['ram_total_gb'],
                  'available_storage': users_demand['storage_total_gb'],
                  'available_vcpu': users_demand['cpu_total_cores'],
                  'available_gpu': users_demand['gpu_total_tflops'],
                  'available_tpu': users_demand['tpu_total_tops'],
              },
              'max_distance': topology_request['max_distance'],
          }
  
  pricing_path = pdsa.generators.pricing_from_topology(
                    topology_id=topology_id,
                    topologies_result_dir=TOPOLOGIES_DIR,
                    compatible_provider_groups=pdsa.generators.compatible_provider_groups_from_offer(topology_offer),
                    options={"logs": False}
                )
  
  pricing_obj = pdsa.utils.yaml_to_pricing_proto(
      os.path.join(TOPOLOGIES_DIR, topology_id, "pricing.yml"), 
      Pricing
  )
  
  problem_instance_pricing, filter_criteria = pdsa.generators.problem_instance(
                                                  instance_pricing=pricing_obj,
                                                  request=request,
                                                  topologies_result_dir=TOPOLOGIES_DIR,
                                                  unlimited_value=UNLIMITED_VALUE
                                              )
  
  pdsa.utils.pricing_proto_to_yaml(
    problem_instance_pricing, 
    os.path.join(TOPOLOGIES_DIR, topology_id, "problem_instance_pricing.yml"),
    options={"logs": False}
  )
  
  filters[id] = filter_criteria
  

---- Generating problem instance for scenario small_ar/vr ----
----- Demand -----
{'service_mode': <AppType.AR_VR: 'ar_vr'>, 'active_users': 3324, 'ram_total_gb': 26616, 'gpu_total_tflops': 13265, 'cpu_total_cores': 6602, 'storage_total_gb': 5021, 'tpu_total_tops': 733}
---- Generating problem instance for scenario medium_ar/vr ----
----- Demand -----
{'service_mode': <AppType.AR_VR: 'ar_vr'>, 'active_users': 3324, 'ram_total_gb': 26616, 'gpu_total_tflops': 13265, 'cpu_total_cores': 6602, 'storage_total_gb': 5021, 'tpu_total_tops': 733}
---- Generating problem instance for scenario large_ar/vr ----
----- Demand -----
{'service_mode': <AppType.AR_VR: 'ar_vr'>, 'active_users': 3324, 'ram_total_gb': 26616, 'gpu_total_tflops': 13265, 'cpu_total_cores': 6602, 'storage_total_gb': 5021, 'tpu_total_tops': 733}


# 5. Pricing Optimization

In [10]:
for id in topologies.keys():
  instance_path = os.path.join(TOPOLOGIES_DIR, topologies[id], "problem_instance_pricing.yml")
  # instance = ""
  # with open(instance_path, 'r') as f:
  #   instance = f.read()
    
  # print(instance)
  
  filter = filters[id]
  
  result = pdsa.optimize(PRIME_INSTANCE_URL, instance_path, filter)
  
  results[id] = result
  
  

In [11]:
print(results["small_ar/vr"])

{'jobId': 'job-4d757ce8-f92b-45c6-b4c2-9b481c933698', 'status': 'COMPLETED', 'submittedAt': '2026-02-16T16:55:35.867Z', 'startedAt': '2026-02-16T16:55:35.867Z', 'completedAt': '2026-02-16T16:55:36.341Z', 'result': {'error': 'No solution found for the given filters.'}}
